In [3]:
import random
import csv
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple

In [4]:
@dataclass
class Restaurant:
    cuisines:list
    price_range:str
    has_self_del:bool
    has_offer:bool
    has_extra_del_cost:bool
    min_cost:float
    avg_rating:float
    avg_del_time:int
    payment_methods:list


In [5]:


def generate_restaurants(
    rest_num: int = 100,
    cuisine_num_max: int = 4,
    cuisine_options: List[str] =  ['Pizza', 'Burger', 'Pasta', 'Souvlaki', 'Sushi', 'Chinese'],
    price_options: List[str] = ["$", "$$", "$$$", "$$$$"],
    price_options_distro: List[float] =  [0.5, 0.3, 0.15, 0.05],
    extra_del_cost_prob: float = 0.2,
    min_cost_gaussian_params: Tuple[float, float] = (6, 3),
    avg_rating_gaussian_params: Tuple[float, float] = (3, 1),
    avg_del_time_gaussian_params: Tuple[float, float] = (30, 15),
    payment_methods: Tuple[str, ...] = ('CASH', 'CARD', 'COUPON')
) -> List[Restaurant]:
    """
    Generates a list of restaurants with randomized attributes.

    Args:
    - rest_num (int): Number of restaurants to generate (default: 100).
    - cuisine_num_max (int): Maximum number of cuisines per restaurant (default: 4).
    - cuisine_options (List[str]): Available cuisines (default: ['Pizza', 'Burger', 'Pasta', 'Souvlaki', 'Sushi', 'Chinese']).
    - price_options (List[str]): Price range options (default: ["$", "$$", "$$$", "$$$$"]).
    - price_options_distro (List[float]): Probability distribution of price options (default: [0.5, 0.3, 0.15, 0.05]).
    - extra_del_cost_prob (float): Probability of extra delivery cost (default: 0.2).
    - min_cost_gaussian_params (Tuple[float, float]): Mean and std deviation for minimum order cost (default: (6,3)).
    - avg_rating_gaussian_params (Tuple[float, float]): Mean and std deviation for average rating (default: (3,1)).
    - avg_del_time_gaussian_params (Tuple[float, float]): Mean and std deviation for average delivery time (default: (30,15)).
    - payment_methods (Tuple[str, ...]): Available payment methods (default: ('CASH', 'CARD', 'COUPON')).

    Returns:
    - List[Restaurant]: A list of generated restaurant objects.
    """


    restaurants = []

    for _ in range(rest_num):
        # Pick a random number of cuisines
        cuisine_num = random.randint(1, cuisine_num_max)
        cuisines = random.sample(cuisine_options, cuisine_num)

        # Sample price range based on probability distribution
        price_index = np.random.choice(len(price_options), p=price_options_distro)
        price_range = price_options[price_index]

        # Random attributes
        has_self_del = random.choice([True, False])
        has_offer = random.choice([True, False])
        has_extra_del_cost = random.random() < extra_del_cost_prob

        # Sample Gaussian-distributed values (ensuring non-negative)
        min_cost = round(max(0, random.gauss(*min_cost_gaussian_params)), 1)
        avg_rating = round(max(0, random.gauss(*avg_rating_gaussian_params)), 1)
        avg_del_time = round(max(0, random.gauss(*avg_del_time_gaussian_params)), 1)

        # Sample payment methods
        selected_payment_methods = random.sample(payment_methods, random.randint(1, len(payment_methods)))

        # Create restaurant object
        restaurants.append(
            Restaurant(
                cuisines=cuisines,
                price_range=price_range,
                has_self_del=has_self_del,
                has_offer=has_offer,
                has_extra_del_cost=has_extra_del_cost,
                min_cost=min_cost,
                avg_rating=avg_rating,
                avg_del_time=avg_del_time,
                payment_methods=selected_payment_methods
            )
        )

    return restaurants


In [6]:
restaurants=generate_restaurants(rest_num=100)

In [7]:
restaurants[2]

Restaurant(cuisines=['Sushi'], price_range='$', has_self_del=False, has_offer=False, has_extra_del_cost=False, min_cost=7.5, avg_rating=4.3, avg_del_time=12.4, payment_methods=['CASH', 'CARD', 'COUPON'])

In [8]:
restaurants[2].avg_del_time

12.4

In [9]:
@dataclass
class User:
    gender:str
    age:int
    favorite_cuisines:list


In [10]:


def generate_users_segment1(user_num: int = 1000) -> List[User]:
    """
    Generates a list of users for Segment 1: One-Trick Pony.

    Characteristics:
    - 50% Male, 50% Female (randomly assigned)
    - Single favorite cuisine
    - Age follows a Gaussian distribution with a mean of 60 and std deviation of 7

    Args:
    - user_num (int): Number of users to generate (default: 1000)

    Returns:
    - List[User]: A list of generated users.
    """

    available_cuisines = ['Pizza', 'Burger', 'Pasta', 'Souvlaki', 'Sushi', 'Chinese']

    users = [
        User(
            gender=random.choice(["M", "F"]),
            age=max(0, int(random.gauss(60, 7))),  # Ensure non-negative age
            favorite_cuisines=random.sample(available_cuisines, 1)  # Single favorite cuisine
        )
        for _ in range(user_num)
    ]

    return users


In [11]:

import csv
import random
from typing import List

def generate_ratings_segment1(users: List[User], restaurants: List[Restaurant], output_file: str = 'segment1.csv') -> None:
    """
    Generates ratings for Segment 1: One-Trick Pony.

    Criteria:
    - AVG_RATING > 4
    - AVG_DEL_TIME <= 35 min
    - HAS_OFFER: 60% influence on rating
    - 90% consistent rating behavior

    Args:
    - users (List[User]): List of user objects.
    - restaurants (List[Restaurant]): List of restaurant objects.
    - output_file (str): Name of the CSV file to save ratings (default: 'segment1.csv').

    Returns:
    - None (Writes data to CSV file).
    """

    with open(output_file, 'w', newline='') as fw:
        writer = csv.writer(fw)
        writer.writerow([
            'age', 'gender', 'favorite_cuisines', 'restaurant_cuisines', 'price_range',
            'has_self_del', 'has_offer', 'has_extra_del_cost', 'min_cost',
            'avg_rating', 'avg_del_time', 'payment_methods', 'rating', 'reason'
        ])

        positive_ratings = 0
        total_ratings = 0

        for user in users:
            rating_count = random.randint(1, len(restaurants))
            total_ratings += rating_count
            sampled_restaurants = random.sample(restaurants, rating_count)

            for restaurant in sampled_restaurants:
                rating = -1  # Default: negative rating
                reason = ""

                # Check if the restaurant matches user's single favorite cuisine and meets criteria
                if user.favorite_cuisines[0] in restaurant.cuisines:
                    if restaurant.avg_rating > 4 and restaurant.avg_del_time <= 35:
                        # 50-50 chance or boosted by HAS_OFFER (60% probability)
                        if random.random() < 0.5 or (restaurant.has_offer and random.random() <= 0.6):
                            rating = 1
                            positive_ratings += 1
                            reason = "Good rating & fast delivery"
                            if restaurant.has_offer:
                                reason += " + Offer available"
                        else:
                            reason = "Did not meet offer or random condition"
                    else:
                        reason = "Rating too low or delivery too slow"
                else:
                    reason = "Not preferred cuisine"

                # 90% consistency: 10% chance of flipping rating
                if random.random() < 0.1:
                    rating *= -1
                    reason += " Rating was flipped due to insconsistnet behavior"

                writer.writerow([
                    user.age, user.gender, user.favorite_cuisines, restaurant.cuisines, restaurant.price_range,
                    restaurant.has_self_del, restaurant.has_offer, restaurant.has_extra_del_cost,
                    restaurant.min_cost, restaurant.avg_rating, restaurant.avg_del_time,
                    restaurant.payment_methods, rating, reason
                ])

    print(f'Positive Ratings: {positive_ratings / total_ratings:.2%}')


In [12]:
users_segment1=generate_users_segment1(user_num=1000)
generate_ratings_segment1(users_segment1,restaurants)

Positive Ratings: 1.71%


In [13]:


def generate_users_segment2(user_num: int = 1000, postcode_num: int = 50) -> List[User]:
    """
    Generates a list of users for Segment 2: young and price-driven.

    Characteristics:
    - 50% Male, 50% Female (randomly assigned)
    - Multiple favorite cuisines (at least 2)
    - Age follows a Gaussian distribution with a mean of 20 and std deviation of 7

    Args:
    - user_num (int): Number of users to generate (default: 1000)
    - postcode_num (int): Number of postcodes to generate (not used in function) (default: 50)

    Returns:
    - List[User]: A list of generated users.
    """

    available_cuisines = ['Pizza', 'Burger', 'Pasta', 'Souvlaki', 'Sushi', 'Chinese']

    users = [
        User(
            gender=random.choice(["M", "F"]),
            age=max(0, int(random.gauss(20, 7))),  # Ensure non-negative age
            favorite_cuisines=random.sample(available_cuisines, random.randint(2, 6))
        )
        for _ in range(user_num)
    ]

    return users


In [14]:

def generate_ratings_segment2(users: List[User], restaurants: List[Restaurant], output_file: str = 'segment2.csv') -> None:
    """
    Generates ratings for Segment 2: young and price-driven.

    Criteria:
    - AVG_RATING > 4.2 if price is $$$
    - AVG_RATING > 3.5 if price is $$
    - AVG_RATING > 3 if price is $
    - No ratings if price is $$$$
    - No ratings if restaurant has an extra delivery cost
    - HAS_OFFER boosts rating probability: 80% influence for $$$, 60% influence for $, $$
    - 80% consistency in rating behavior

    Args:
    - users (List[User]): List of user objects.
    - restaurants (List[Restaurant]): List of restaurant objects.
    - output_file (str): Name of the CSV file to save ratings (default: 'segment2.csv').
    """

    with open(output_file, 'w', newline='') as fw:
        writer = csv.writer(fw)
        writer.writerow([
            'age', 'gender', 'favorite_cuisines', 'restaurant_cuisines', 'price_range',
            'has_self_del', 'has_offer', 'has_extra_del_cost', 'min_cost',
            'avg_rating', 'avg_del_time', 'payment_methods', 'rating', 'reason'
        ])

        positive_ratings = 0
        total_ratings = 0

        for user in users:
            num_ratings = random.randint(1, len(restaurants))
            total_ratings += num_ratings
            sampled_restaurants = random.sample(restaurants, num_ratings)

            for restaurant in sampled_restaurants:
                rating = -1  # Default negative rating
                reason = ""

                if any(cuisine in restaurant.cuisines for cuisine in user.favorite_cuisines):
                    if restaurant.has_extra_del_cost:
                        reason = "Extra delivery cost"
                    else:
                        if (
                            (restaurant.price_range == '$' and restaurant.avg_rating > 3) or
                            (restaurant.price_range == '$$' and restaurant.avg_rating > 3.5) or
                            (restaurant.price_range == '$$$' and restaurant.avg_rating > 4.2)
                        ):
                            # Rating decision logic with offer influence
                            if (
                                random.random() < 0.5 or
                                (restaurant.has_offer and restaurant.price_range in ['$', '$$'] and random.random() <= 0.6) or
                                (restaurant.has_offer and restaurant.price_range == '$$$' and random.random() <= 0.8)
                            ):
                                rating = 1
                                positive_ratings += 1
                                reason = "Good price-quality balance"
                                if restaurant.has_offer:
                                    reason += " + Offer available"
                            else:
                                reason = "Did not meet offer or price conditions"
                        else:
                            reason = "Did not meet rating and price conditions"
                else:
                    reason = "Not preferred cuisine"

                # 80% consistency factor (flip rating 20% of the time)
                if random.random() < 0.2:
                    rating *= -1
                    reason += " Inconsistent rating behavior"


                writer.writerow([
                    user.age, user.gender, user.favorite_cuisines, restaurant.cuisines, restaurant.price_range,
                    restaurant.has_self_del, restaurant.has_offer, restaurant.has_extra_del_cost,
                    restaurant.min_cost, restaurant.avg_rating, restaurant.avg_del_time,
                    restaurant.payment_methods, rating, reason
                ])

    print(f'Positive Ratings: {positive_ratings/total_ratings:.2%} ({positive_ratings}/{total_ratings})')



In [15]:
users_segment2=generate_users_segment2(user_num=2000)
generate_ratings_segment2(users_segment2,restaurants)

Positive Ratings: 16.01% (16050/100270)


In [16]:
file_paths = ['segment1.csv', 'segment2.csv']  # Add as many paths as needed

# Read each CSV file into a DataFrame and store them in a list
dfs = [pd.read_csv(file_path) for file_path in file_paths]

# Concatenate all DataFrames into a single DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

# Shuffle the rows of the combined DataFrame
df = combined_df.sample(frac=1).reset_index(drop=True)

# shuffled_df now contains all your data, shuffled
print(df.shape)

(151656, 14)


In [17]:
df

,age,gender,favorite_cuisines,restaurant_cuisines,price_range,has_self_del,has_offer,has_extra_del_cost,min_cost,avg_rating,avg_del_time,payment_methods,rating,reason
0,30,F,"['Chinese', 'Souvlaki', 'Sushi', 'Pasta']","['Souvlaki', 'Sushi']",$$$,False,True,False,4.9,3.7,15.4,"['CASH', 'CARD', 'COUPON']",-1,Did not meet rating and price conditions
1,27,F,"['Souvlaki', 'Pasta', 'Pizza']",['Pizza'],$,False,True,False,4.4,4.8,36.3,"['CARD', 'CASH']",1,Good price-quality balance + Offer available
2,8,M,"['Sushi', 'Pasta', 'Souvlaki', 'Chinese', 'Bur...",['Chinese'],$$,True,False,False,6.7,4.6,1.5,"['CASH', 'COUPON', 'CARD']",-1,Good price-quality balance Inconsistent rating...
3,15,M,"['Sushi', 'Souvlaki', 'Chinese', 'Pasta', 'Piz...","['Pizza', 'Sushi', 'Burger']",$$$,False,True,True,13.1,2.7,9.9,"['CASH', 'CARD']",-1,Extra delivery cost
4,57,F,['Sushi'],"['Souvlaki', 'Chinese', 'Pizza', 'Sushi']",$$,False,False,True,2.1,3.5,25.1,['CARD'],-1,Rating too low or delivery too slow
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151651,56,F,['Chinese'],"['Sushi', 'Chinese', 'Burger', 'Souvlaki']",$$,False,True,False,10.5,5.4,45.3,"['CARD', 'CASH', 'COUPON']",-1,Rating too low or delivery too slow
151652,57,F,['Burger'],"['Burger', 'Pasta', 'Souvlaki', 'Pizza']",$$$,False,False,False,5.2,3.2,36.7,"['COUPON', 'CARD']",-1,Rating too low or delivery too slow
151653,9,F,"['Pizza', 'Pasta', 'Souvlaki', 'Chinese']","['Chinese', 'Sushi', 'Pizza', 'Souvlaki']",$,True,False,False,2.0,3.3,7.3,"['CASH', 'CARD']",1,Did not meet offer or price conditions Inconsi...
151654,36,M,"['Burger', 'Chinese', 'Sushi', 'Pasta', 'Pizza']","['Sushi', 'Burger', 'Pasta', 'Pizza']",$$,True,False,False,4.7,3.1,29.3,['COUPON'],-1,Did not meet rating and price conditions
